# 02. Limpieza y Enriquecimiento de Datos (Feature Engineering)

Una vez que entendieron los datos, vamos a prepararlos para que los algoritmos de Machine Learning puedan procesarlos correctamente. En la vida real, lo mejor es empaquetar esto automatizado en Scikit-Learn Pipelines o en scripts de Python (`src/features/build_features.py`). Aquí puedes experimentar con las rutinas de limpieza.

### Instrucciones Generales:
1. **Solvertar problema de calidad:**: Solucionar problema de calidad encontrados en el EDA: consistencia, sensibilidad, precision y completitud. Documenta cada decision tomada.
2. **Codificación Categórica:** El campo `ocean_proximity` es de texto. Conviértelo en variable numerica, ya que los algoritmos clasicos no entienen el texto. Documenta porque usaste codificacion Ordinal o Nominal.
3. **Enriquecimiento (Feature Engineering):** Como pudiste notar en tu análisis, `total_rooms` no significa mucho si hay muchos hogares en un distrito. Agrega nuevas métricas útiles, por ejemplo:
   - `rooms_per_household = total_rooms / households`
   - `bedrooms_per_room = total_bedrooms / total_rooms`
   - `population_per_household = population / households`
4. **Escalado de Variables:** Aplica un `StandardScaler` o `MinMaxScaler` para evitar que las variables numéricas grandes pesen más en algoritmos basados en distancias o gradientes.


In [1]:
# Escribe tu código aquí para limpiar y enriquecer los datos
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler


In [2]:
df = pd.read_csv("../data/raw/housing/housing.csv")
print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head()

Dimensiones: 20640 filas x 10 columnas


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [3]:
#Calidad
print("=== VALORES NULOS ANTES ===")
print(df.isnull().sum())

# Imputar total_bedrooms con la mediana (robusta a outliers)
imputer = SimpleImputer(strategy="median")
df["total_bedrooms"] = imputer.fit_transform(df[["total_bedrooms"]])

print("\n=== VALORES NULOS DESPUÉS ===")
print(df.isnull().sum())
print("\n✅ Decisión: Se imputó total_bedrooms con la MEDIANA porque es robusta a outliers")

=== VALORES NULOS ANTES ===
longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

=== VALORES NULOS DESPUÉS ===
longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0
ocean_proximity       0
dtype: int64

✅ Decisión: Se imputó total_bedrooms con la MEDIANA porque es robusta a outliers


In [4]:
#Eliminar las 8 inconsistencias
# Eliminarlas porque son errores reales
indices_malos = df[
    (df["population"] < df["households"]) |
    (df["total_rooms"] / df["households"] < 1)
].index

df_limpio = df.drop(indices_malos)
print(f"Filas eliminadas: {len(indices_malos)}")
print(f"Filas restantes: {len(df_limpio)}")

Filas eliminadas: 5
Filas restantes: 20635


In [5]:
#Codificación categórica
print("Valores únicos de ocean_proximity:")
print(df["ocean_proximity"].value_counts())

# Usamos OneHotEncoder (Nominal) porque las categorías NO tienen orden natural
# Ordinal implicaría que INLAND > NEAR BAY > NEAR OCEAN, lo cual es incorrecto
ohe = OneHotEncoder(sparse_output=False)
ocean_encoded = ohe.fit_transform(df[["ocean_proximity"]])
ocean_df = pd.DataFrame(ocean_encoded, columns=ohe.get_feature_names_out(["ocean_proximity"]))

# Unir al dataframe y eliminar columna original
df = df.drop("ocean_proximity", axis=1).reset_index(drop=True)
df = pd.concat([df, ocean_df], axis=1)

print("\n✅ Decisión: Se usó OneHotEncoder (Nominal) porque las categorías no tienen orden")
print(f"Nuevas columnas: {list(ohe.get_feature_names_out(['ocean_proximity']))}")
df.head()

Valores únicos de ocean_proximity:
ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5
Name: count, dtype: int64

✅ Decisión: Se usó OneHotEncoder (Nominal) porque las categorías no tienen orden
Nuevas columnas: ['ocean_proximity_<1H OCEAN', 'ocean_proximity_INLAND', 'ocean_proximity_ISLAND', 'ocean_proximity_NEAR BAY', 'ocean_proximity_NEAR OCEAN']


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity_<1H OCEAN,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,0.0,0.0,0.0,1.0,0.0
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,0.0,0.0,0.0,1.0,0.0
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,0.0,0.0,0.0,1.0,0.0
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,0.0,0.0,0.0,1.0,0.0
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,0.0,0.0,0.0,1.0,0.0


In [6]:
#Feature Engineering
df["rooms_per_household"]       = df["total_rooms"]    / df["households"]
df["bedrooms_per_room"]         = df["total_bedrooms"] / df["total_rooms"]
df["population_per_household"]  = df["population"]     / df["households"]

print("✅ Nuevas variables creadas:")
print(df[["rooms_per_household", "bedrooms_per_room", "population_per_household"]].describe())

✅ Nuevas variables creadas:
       rooms_per_household  bedrooms_per_room  population_per_household
count         20640.000000       20640.000000              20640.000000
mean              5.429000           0.213794                  3.070655
std               2.474173           0.065248                 10.386050
min               0.846154           0.037151                  0.692308
25%               4.440716           0.175225                  2.429741
50%               5.229129           0.203159                  2.818116
75%               6.052381           0.240126                  3.282261
max             141.909091           2.824675               1243.333333


In [7]:
print(df.columns.tolist())

['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value', 'ocean_proximity_<1H OCEAN', 'ocean_proximity_INLAND', 'ocean_proximity_ISLAND', 'ocean_proximity_NEAR BAY', 'ocean_proximity_NEAR OCEAN', 'rooms_per_household', 'bedrooms_per_room', 'population_per_household']


In [8]:
#Correlaciones después del Feature Engineering
correlaciones = df.corr(numeric_only=True)["median_house_value"].sort_values(ascending=False)
print("=== CORRELACIONES ACTUALIZADAS ===")
print(correlaciones)

=== CORRELACIONES ACTUALIZADAS ===
median_house_value            1.000000
median_income                 0.688075
ocean_proximity_<1H OCEAN     0.256617
ocean_proximity_NEAR BAY      0.160284
rooms_per_household           0.151948
ocean_proximity_NEAR OCEAN    0.141862
total_rooms                   0.134153
housing_median_age            0.105623
households                    0.065843
total_bedrooms                0.049457
ocean_proximity_ISLAND        0.023416
population_per_household     -0.023737
population                   -0.024650
longitude                    -0.045967
latitude                     -0.144160
bedrooms_per_room            -0.233303
ocean_proximity_INLAND       -0.484859
Name: median_house_value, dtype: float64


In [ ]:
#Escalado de variables
# Columnas a escalar (solo numéricas originales)
cols_escalar = ["longitude", "latitude", "housing_median_age", "total_rooms",
                "total_bedrooms", "population", "households", "median_income",
                "rooms_per_household", "bedrooms_per_room", "population_per_household"]

scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[cols_escalar] = scaler.fit_transform(df[cols_escalar])

print("✅ Escalado aplicado con StandardScaler")
print("\n✅ Decisión: StandardScaler porque los modelos basados en gradientes (SGD)")
print("   y distancias son sensibles a la magnitud de las variables")
df_scaled.head()

✅ Escalado aplicado con StandardScaler

✅ Decisión: StandardScaler porque los modelos basados en gradientes (SGD)
   y distancias son sensibles a la magnitud de las variables


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity_<1H OCEAN,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN,rooms_per_household,bedrooms_per_room,population_per_household
0,-1.327835,1.052548,0.982143,-0.804819,-0.972476,-0.974429,-0.977033,2.344766,452600.0,0.0,0.0,0.0,1.0,0.0,0.628559,-1.029988,-0.049597
1,-1.322844,1.043185,-0.607019,2.045890,1.357143,0.861439,1.669961,2.332238,358500.0,0.0,0.0,0.0,1.0,0.0,0.327041,-0.888897,-0.092512
2,-1.332827,1.038503,1.856182,-0.535746,-0.827024,-0.820777,-0.843637,1.782699,352100.0,0.0,0.0,0.0,1.0,0.0,1.155620,-1.291686,-0.025843
3,-1.337818,1.038503,1.856182,-0.624215,-0.719723,-0.766028,-0.733781,0.932968,341300.0,0.0,0.0,0.0,1.0,0.0,0.156966,-0.449613,-0.050329
4,-1.337818,1.038503,1.856182,-0.462404,-0.612423,-0.759847,-0.629157,-0.012881,342200.0,0.0,0.0,0.0,1.0,0.0,0.344711,-0.639087,-0.085616


In [10]:
#Guardar datos procesados
import os
os.makedirs("../data/interim", exist_ok=True)
df_scaled.to_csv("../data/interim/train_clean.csv", index=False)
print("✅ Datos limpios guardados en data/interim/train_clean.csv")

✅ Datos limpios guardados en data/interim/train_clean.csv


## Conclusiones de Limpieza y Feature Engineering

1. **Imputación:** Se rellenaron ~207 valores nulos en `total_bedrooms` 
   con la mediana, evitando pérdida de registros.

2. **Codificación:** Se usó OneHotEncoder para `ocean_proximity` porque 
   sus categorías no tienen orden natural (Nominal).

3. **Nuevas variables:** Las métricas por hogar (`rooms_per_household`, 
   `bedrooms_per_room`) capturan mejor la realidad del distrito que 
   los totales absolutos.

4. **Escalado:** StandardScaler normaliza todas las variables numéricas 
   para que algoritmos como SGD y KNN no sean afectados por magnitudes grandes.